# Lezione 12 — Governance, costi e LLMOps + progetto

**Corso:** MLOps e Monitoraggio con Python · **Blocco B** · **ULTIMA lezione**
**Tempo stimato:** 120 minuti · **Capitolo del manuale:** `manual/12_gobernanza_costes_llmops_IT.md`

## Cosa vedrai in questa lezione

- **Governance:** un piccolo registro di audit che unisce DVC (lezione 3), MLflow (lezione 4) e il registro dei modelli con alias (lezione 8) in un report leggibile di tracciabilità completa.
- **Costi:** una stima REALE, calcolata in Python, del costo di addestramento, inferenza (basata sulla latenza reale della lezione 7) e storage (con dimensioni reali di file) di un sistema in produzione.
- **LLMOps:** in cosa si assomiglia e in cosa si differenzia dall'MLOps classico di questo corso, con una catena RAG simulata (senza nessun modello di linguaggio reale) tracciata con `mlflow.trace`.
- **Progetto integratore:** un caso nuovo e più piccolo (cancro al seno, `sklearn.datasets.load_breast_cancer`) percorrendo con criterio le decisioni delle lezioni 2-11.

> **Nota:** tutto avviene dentro una cartella di lavoro effimera (`proyecto_gobernanza/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

> **Attenzione:** tutte le cifre di costo sono un *esempio illustrativo*, con costanti dichiarate esplicitamente (costo per ora di CPU, costo per GB, costo per 1000 token). Il punto pedagogico è la formula e il ragionamento, non il numero assoluto.

## Prerequisiti

- Dalla lezione 3: lineage di dati e modelli con DVC (hash, commit).
- Dalla lezione 4: tracciamento degli esperimenti con MLflow.
- Dalla lezione 8: registro dei modelli con alias candidato/produzione/archiviato.
- Dalla lezione 7: la latenza reale misurata del servizio di predizione.
- Dalle lezioni 2, 6, 10: seed e config esplicita, test minimi, controllo di soglie.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (incluso MLflow e joblib).

In [ ]:
import os
import warnings

# Silenziamo avvisi di compatibilità fissi (non aggiungono nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Come nelle lezioni 4 e 8: fissiamo queste variabili PRIMA di importare mlflow e alziamo il suo logger a ERROR
# (WARNING non basta: mlflow.utils.requirements_utils avvisa in WARNING con dettagli non deterministici)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("mlflow", "mlflow")
print("Librerie pronte.")

---
## Cella 0 — Setup

In [ ]:
import os
import re
import shutil
import json
import logging
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.datasets import load_breast_cancer

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

import mlflow

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerie e funzioni ausiliarie pronte. Versione di MLflow:", mlflow.__version__)

---
## Cartella di lavoro effimera

In [ ]:
CARPETA = "proyecto_gobernanza"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero (come in tutte le lezioni precedenti)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Governance: un registro di audit di tracciabilità completa

Uniamo, in un unico report, ciò che già esiste separatamente: il lineage di dati e modelli (lezione 3, hash di DVC), il tracciamento degli esperimenti (lezione 4, `run_id` di MLflow) e il registro dei modelli con alias (lezione 8, candidato/produzione/archiviato). Gli eventi qui sotto sono **simulati** (non un'esecuzione reale di quegli strumenti), ma la funzione che li elabora è codice reale.

In [ ]:
# Eventi SIMULATI (dati di esempio, non un'esecuzione reale di DVC/MLflow/registro): uniscono concettualmente
# il lineage dei dati (lezione 3), il tracciamento degli esperimenti (lezione 4) e il registro dei modelli con alias (lezione 8)
EVENTOS = [
    {"tipo": "version_datos", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-10",
     "detalle": "hash_dvc=a1b2c3d4 (churn.csv, clase 3)"},
    {"tipo": "entrenamiento", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-11",
     "detalle": "run_mlflow=r001, class_weight=ninguno, f1=0.02 (clase 4)"},
    {"tipo": "promocion_candidato", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-11",
     "detalle": "alias candidato -> v1 (clase 8)"},
    {"tipo": "aprobacion_produccion", "version_modelo": "v1", "actor": "luis_revisor", "cuando": "2026-01-12",
     "detalle": "alias produccion -> v1, tests superados (clase 6)"},
    {"tipo": "version_datos", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-03",
     "detalle": "hash_dvc=e5f6a7b8 (churn.csv v2, clase 3)"},
    {"tipo": "entrenamiento", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-04",
     "detalle": "run_mlflow=r002, class_weight=balanced, f1=0.3166 (clase 4)"},
    {"tipo": "promocion_candidato", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-04",
     "detalle": "alias candidato -> v2 (clase 8)"},
    {"tipo": "aprobacion_produccion", "version_modelo": "v2", "actor": "luis_revisor", "cuando": "2026-02-05",
     "detalle": "alias produccion -> v2, alias archivado -> v1 (clase 8)"},
]
print("Eventi di audit caricati:", len(EVENTOS))

In [ ]:
def generar_informe_auditoria(eventos):
    """Raggruppa gli eventi per versione di modello e produce un report leggibile di tracciabilità completa:
    chi ha addestrato, chi ha promosso, chi ha approvato -- rispondendo "chi ha fatto cosa e quando" per ogni versione."""
    versiones = sorted({e["version_modelo"] for e in eventos})
    lineas = []
    for version in versiones:
        eventos_version = [e for e in eventos if e["version_modelo"] == version]
        lineas.append(f"Versione {version}:")
        for e in sorted(eventos_version, key=lambda x: x["cuando"]):
            lineas.append(f"  [{e['cuando']}] {e['tipo']} -- {e['actor']} :: {e['detalle']}")
    return "\n".join(lineas)


def verificar_separacion_de_funciones(eventos):
    """Principio di controllo degli accessi (separazione dei compiti): chi ADDESTRA una versione non dovrebbe
    essere la STESSA persona che la APPROVA per la produzione. Restituisce la lista di versioni che violano questa regola."""
    violaciones = []
    for version in {e["version_modelo"] for e in eventos}:
        entrenadores = {e["actor"] for e in eventos if e["version_modelo"] == version and e["tipo"] == "entrenamiento"}
        aprobadores = {e["actor"] for e in eventos if e["version_modelo"] == version and e["tipo"] == "aprobacion_produccion"}
        if entrenadores & aprobadores:
            violaciones.append(version)
    return violaciones


informe = generar_informe_auditoria(EVENTOS)
print(informe)

RUTA_INFORME = os.path.join(CARPETA, "informe_auditoria.txt")
with open(RUTA_INFORME, "w") as f:
    f.write(informe)
print("\nReport scritto in:", RUTA_INFORME)

violaciones = verificar_separacion_de_funciones(EVENTOS)
print("Versioni che violano la separazione dei compiti (chi addestra == chi approva):", violaciones)
assert violaciones == [], "il registro simulato non dovrebbe avere violazioni di separazione dei compiti"
print("Separazione dei compiti rispettata: nessuno ha approvato se stesso.")

**Cosa osservi?** Il report risponde, per ogni versione, "chi ha addestrato", "chi ha promosso a candidato" e "chi ha approvato la produzione" -- esattamente i tre elementi di tracciabilità completa che il capitolo descrive. La funzione `verificar_separacion_de_funciones` codifica un principio di **controllo degli accessi** (sezione 2 del capitolo): chi addestra non dovrebbe essere chi approva il proprio lavoro per la produzione, indipendentemente da quale provider cloud o strumento concreto si usi.

---
## 2. Costi: una stima reale, calcolata in Python

Tre componenti, ciascuno con la sua formula esplicita e le sue costanti dichiarate come *esempio illustrativo*.

In [ ]:
# 2.1 Costo di addestramento: ore di calcolo ILLUSTRATIVE x un costo orario ILLUSTRATIVO
# (l'addestramento reale di questo progetto richiede secondi su un portatile: le 2 ore simulano un caso di
# produzione più grande, con più dati e validazione incrociata -- il PUNTO è la formula, non la cifra assoluta)
HORAS_ENTRENAMIENTO_ILUSTRATIVO = 2.0
COSTO_HORA_CPU_ILUSTRATIVO = 0.05  # USD/ora, esempio illustrativo, NON legato a un fornitore concreto (i prezzi cambiano)

costo_entrenamiento = round(HORAS_ENTRENAMIENTO_ILUSTRATIVO * COSTO_HORA_CPU_ILUSTRATIVO, 4)
print("Costo di addestramento (esempio illustrativo):", costo_entrenamiento, "USD")

# 2.2 Costo di inferenza: si appoggia sulla latenza REALE misurata nella lezione 7 (lì, ≈4 ms per richiesta,
# valore usato anche nella sua domanda di autovalutazione 2) -- qui la fissiamo come costante perché il
# calcolo sia riproducibile byte per byte (la lezione 7 aveva già dichiarato che il millisecondo esatto varia per macchina)
LATENCIA_MS_ILUSTRATIVA = 4.0  # stesso ordine di grandezza della latenza reale misurata nella lezione 7

costo_por_segundo = COSTO_HORA_CPU_ILUSTRATIVO / 3600
costo_por_prediccion = costo_por_segundo * (LATENCIA_MS_ILUSTRATIVA / 1000)
costo_por_1000_predicciones = round(costo_por_prediccion * 1000, 6)
print("Costo per 1000 predizioni (esempio illustrativo):", costo_por_1000_predicciones, "USD")

PREDICCIONES_MES_ILUSTRATIVO = 100_000
costo_inferencia_mensual = round(costo_por_1000_predicciones * (PREDICCIONES_MES_ILUSTRATIVO / 1000), 4)
print("Costo di inferenza mensile, per", PREDICCIONES_MES_ILUSTRATIVO, "predizioni/mese (esempio illustrativo):", costo_inferencia_mensual, "USD")

In [ ]:
# 2.3 Costo di storage: usiamo la dimensione REALE (misurata con os.path.getsize, non indovinata) del dataset
# (lo stesso churn.csv delle lezioni 3-11) e dell'artefatto addestrato, x un costo per GB ILLUSTRATIVO
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)
X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)

preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)
F1 = round(f1_score(y_test, pipeline.predict(X_test)), 4)
print("F1 del modello riaddestrato (config. vincitrice delle lezioni 4/5/7/8):", F1)

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)

tamano_dataset_bytes = os.path.getsize(RUTA_CSV)
tamano_modelo_bytes = os.path.getsize(RUTA_MODELO)
tamano_total_gb = (tamano_dataset_bytes + tamano_modelo_bytes) / 1024 ** 3
print("Dimensione reale di churn.csv (byte):", tamano_dataset_bytes)
print("Dimensione reale di modelo.pkl (byte):", tamano_modelo_bytes)

COSTO_POR_GB_MES_ILUSTRATIVO = 0.02  # USD/GB/mese, esempio illustrativo
costo_almacenamiento_mensual = round(tamano_total_gb * COSTO_POR_GB_MES_ILUSTRATIVO, 8)
print("Costo di storage mensile (esempio illustrativo):", costo_almacenamiento_mensual, "USD")

costo_mensual_total = round(costo_entrenamiento + costo_inferencia_mensual + costo_almacenamiento_mensual, 4)
print("Costo TOTALE del primo mese (addestramento unico + inferenza + storage, esempio illustrativo):", costo_mensual_total, "USD")

**Cosa osservi?** Il costo di addestramento (fisso, avviene una volta) domina il primo mese di questo piccolo esempio; il costo di inferenza scala con il volume di predizioni (qui, 100 000/mese illustrativo); il costo di storage è praticamente irrilevante per un dataset e un modello di questa dimensione (circa 154 KB in totale) -- sarebbe molto diverso con dataset di svariati GB o modelli di miliardi di parametri, come quelli della sezione 3.

---
## 3. LLMOps: una catena RAG simulata, tracciata con MLflow

Nessuna chiamata a un modello di linguaggio reale: "recuperare" e "generare" sono funzioni deterministiche su una lista fissa di 3 documenti. Le tracciamo con `@mlflow.trace`, lo stesso file store locale delle lezioni 4 e 8 (logger a `ERROR`, senza server).

In [ ]:
# NON usiamo nessun modello di linguaggio reale (senza API, senza download): "recupero" e "generazione" sono
# funzioni deterministiche su una lista fissa di documenti, per poterle tracciare con MLflow senza dipendere
# da nessun servizio esterno
DOCUMENTOS = [
    {"id": "doc1", "texto": "El PSI mide el desplazamiento de una distribucion entre dos periodos usando buckets fijos"},  # dato-es
    {"id": "doc2", "texto": "MLflow registra experimentos modelos y ahora tambien trazas de llamadas encadenadas"},  # dato-es
    {"id": "doc3", "texto": "Docker empaqueta un servicio en una imagen reproducible con todas sus dependencias"},  # dato-es
]


def contar_tokens_falso(texto):
    """Conteggio di token FINTO ma coerente (una parola = un token), sufficiente per illustrare il
    principio di costo per token, senza un tokenizzatore reale."""
    return len(texto.split())


@mlflow.trace(span_type="RETRIEVER")
def recuperar(pregunta):
    """Passo 1: "recupera" il documento il cui testo condivide più parole con la domanda (deterministico)."""
    palabras_pregunta = set(pregunta.lower().split())
    mejor = max(DOCUMENTOS, key=lambda d: len(palabras_pregunta & set(d["texto"].lower().split())))
    return mejor


@mlflow.trace(span_type="LLM")
def generar(pregunta, documento):
    """Passo 2: "genera" una risposta con una funzione mock (testo fisso, senza modello reale)."""
    return f"Segun {documento['id']}: {documento['texto']}"


@mlflow.trace(span_type="CHAIN")
def cadena_rag(pregunta):
    """Passo 3: incatena recupero + generazione -- la "catena" che si traccia come un'unità."""
    documento = recuperar(pregunta)
    return generar(pregunta, documento)


RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)
mlflow.set_experiment("rag_simulado")

PREGUNTA = "que mide el PSI"  # dato-es
with mlflow.start_run(run_name="consulta_rag"):
    respuesta = cadena_rag(PREGUNTA)
print("Domanda:", PREGUNTA)
print("Risposta:", respuesta)

### 3.1 Leggere la traccia salvata e mostrarne la struttura (non i suoi tempi)

In [ ]:
# Rileggiamo la traccia salvata (file store locale, stessa cautela delle lezioni 4 e 8: logger a ERROR)
experimento = mlflow.get_experiment_by_name("rag_simulado")
trazas = mlflow.search_traces(experiment_ids=[experimento.experiment_id], flush=True)
assert len(trazas) == 1, "ci si aspettava esattamente una traccia salvata"

traza = mlflow.get_trace(trazas.iloc[0]["trace_id"])
spans = traza.data.spans


def estructura_de_traza(spans):
    """Restituisce la STRUTTURA della traccia (nomi, tipi, annidamento) senza id né tempi assoluti --
    quelli SÌ sono deterministici, a differenza di una durata reale in millisecondi (la stessa categoria di
    non-determinismo già dichiarata nella lezione 7 con la latenza reale)."""
    raiz = next(s for s in spans if s.parent_id is None)
    hijos = [s for s in spans if s.parent_id == raiz.span_id]
    return {
        "n_spans": len(spans),
        "raiz": (raiz.name, raiz.span_type),
        "hijos_de_la_raiz": sorted((h.name, h.span_type) for h in hijos),
    }


ESTRUCTURA = estructura_de_traza(spans)
print("Struttura della traccia (n. di span, nomi, tipi, annidamento -- SENZA tempi assoluti):")
print(ESTRUCTURA)
assert ESTRUCTURA["n_spans"] == 3
assert ESTRUCTURA["raiz"] == ("cadena_rag", "CHAIN")
assert ESTRUCTURA["hijos_de_la_raiz"] == [("generar", "LLM"), ("recuperar", "RETRIEVER")]
print("La catena ha 3 passi: 1 CHAIN radice con 2 figli (RETRIEVER e LLM) -- la stessa forma in qualsiasi esecuzione.")

> **Attenzione (determinismo tra lingue):** una durata reale in millisecondi di questa traccia NON sarebbe identica tra due esecuzioni (la stessa categoria di non-determinismo già dichiarata nella lezione 7 con la latenza reale) -- per questo questa sezione mostra la STRUTTURA della traccia (numero di span, nomi, tipi, annidamento), che invece È identica su qualsiasi macchina e in qualsiasi esecuzione.

### 3.2 Costo per token (illustrativo), non per ora di calcolo

In [ ]:
# 3.3 Costo per token ILLUSTRATIVO sulla catena simulata (costo per token, non per ora di calcolo --
# la differenza chiave dell'LLMOps rispetto all'MLOps classico, sezione 3 del capitolo)
documento_recuperado = recuperar(PREGUNTA)
tokens_entrada = contar_tokens_falso(PREGUNTA) + contar_tokens_falso(documento_recuperado["texto"])
tokens_salida = contar_tokens_falso(respuesta)
tokens_totales = tokens_entrada + tokens_salida
print("Token di ingresso (domanda + documento recuperato):", tokens_entrada)
print("Token di uscita (risposta generata):", tokens_salida)

COSTO_POR_1000_TOKENS_ILUSTRATIVO = 0.002  # USD/1000 token, esempio illustrativo, NON legato a un fornitore concreto
costo_cadena = round((tokens_totales / 1000) * COSTO_POR_1000_TOKENS_ILUSTRATIVO, 8)
print("Costo di questa singola chiamata alla catena (esempio illustrativo):", costo_cadena, "USD")

### 3.3 Valutare il testo generato: una metrica euristica, non un'unica cifra aggregata

In [ ]:
# 3.4 Valutare TESTO generato: non c'è un unico numero come F1 o AUC (lezioni 4-11) che basti da solo.
# Qui usiamo una metrica euristica DETERMINISTICA (copertura di parole chiave attese) solo come proxy
# illustrativo -- un sistema reale ha di solito bisogno di revisione umana o un secondo modello "giudice", non solo questo
def evaluar_respuesta(respuesta_generada, palabras_clave_esperadas):
    """Frazione di parole chiave attese che compaiono nella risposta, tra 0 e 1."""
    palabras_respuesta = {p.strip(".,:;") for p in respuesta_generada.lower().split()}
    encontradas = palabras_clave_esperadas & palabras_respuesta
    return round(len(encontradas) / len(palabras_clave_esperadas), 4)


PALABRAS_CLAVE_ESPERADAS = {"psi", "desplazamiento", "distribucion"}
cobertura = evaluar_respuesta(respuesta, PALABRAS_CLAVE_ESPERADAS)
print("Copertura di parole chiave attese nella risposta:", cobertura)
assert cobertura == 1.0, "la risposta dovrebbe contenere le 3 parole chiave di questo esempio"
print("La risposta copre il 100% delle parole chiave attese -- ma questa metrica NON rileva se il testo è coerente, naturale o corretto in altri aspetti: misura solo una coincidenza lessicale.")

**Cosa osservi?** La traccia ha sempre 3 span con la stessa forma (1 `CHAIN` radice con 2 figli, `RETRIEVER` e `LLM`), indipendentemente da quante volte si esegua -- è la differenza con un tempo reale in millisecondi, che invece cambia. Il costo si misura per TOKEN (ingresso + uscita), non per ora di calcolo riservata. E valutare il testo generato non è un F1 o un AUC (lezioni 4-11): la copertura di parole chiave è solo un proxy euristico, utile per un `assert` in questo notebook, ma insufficiente da sola in un sistema reale, dove serve revisione umana o un modello "giudice".

---
## 4. Progetto integratore: un caso nuovo (cancro al seno)

Percorriamo, condensate, le decisioni principali delle lezioni 2-11 su un dataset NUOVO e più piccolo del churn: `sklearn.datasets.load_breast_cancer` (569 righe, 30 colonne numeriche, 2 classi, licenza libera).

In [ ]:
# 4.1 Config esplicita + seed fisso (lezione 2), su un caso NUOVO e più piccolo: cancro al seno
# (scikit-learn, dominio pubblico, classificazione binaria) -- NON è il progetto churn
CONFIG_PROYECTO_NUEVO = {
    "semilla": 42,
    "test_size": 0.25,
    "modelo": "LogisticRegression",
    "max_iter": 2000,
}

datos_cancer = load_breast_cancer()
X_cancer, y_cancer = datos_cancer.data, datos_cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cancer, y_cancer, test_size=CONFIG_PROYECTO_NUEVO["test_size"],
    random_state=CONFIG_PROYECTO_NUEVO["semilla"], stratify=y_cancer,
)

pipeline_cancer = Pipeline([
    ("escalado", StandardScaler()),
    ("clasificador", LogisticRegression(max_iter=CONFIG_PROYECTO_NUEVO["max_iter"], random_state=CONFIG_PROYECTO_NUEVO["semilla"])),
])
pipeline_cancer.fit(Xc_train, yc_train)
pred_cancer = pipeline_cancer.predict(Xc_test)
proba_cancer = pipeline_cancer.predict_proba(Xc_test)[:, 1]

EXACTITUD_CANCER = round(accuracy_score(yc_test, pred_cancer), 4)
F1_CANCER = round(f1_score(yc_test, pred_cancer), 4)
AUC_CANCER = round(roc_auc_score(yc_test, proba_cancer), 4)
print("Dataset:", datos_cancer.data.shape, "righe x colonne, classi:", list(datos_cancer.target_names))
print("Accuratezza (test):", EXACTITUD_CANCER, "| F1:", F1_CANCER, "| AUC:", AUC_CANCER)

In [ ]:
# 4.2 Un test minimo di DATI e uno di MODELLO (lezione 6), senza bisogno di pandera né di una suite completa
def test_forma_de_los_datos():
    """Test di dati: 569 righe, 30 colonne, esattamente 2 classi -- ciò che questo capitolo si aspetta da questo dataset."""
    assert X_cancer.shape == (569, 30)
    assert set(np.unique(y_cancer)) == {0, 1}


def test_modelo_supera_umbral_minimo():
    """Test di modello: come nella lezione 6, il modello non deve scendere sotto una soglia conosciuta in anticipo."""
    assert F1_CANCER >= 0.90


test_forma_de_los_datos()
test_modelo_supera_umbral_minimo()
print("2 test superati: 1 di dati (forma e classi), 1 di modello (F1 minimo).")

In [ ]:
# 4.3 Una funzione di servizio semplice (lezione 7), senza FastAPI: predecir() con validazione minima
NOMBRES_COLUMNAS_CANCER = list(datos_cancer.feature_names)


def predecir(valores):
    """Riceve una lista di 30 numeri (stesso ordine di NOMBRES_COLUMNAS_CANCER) e restituisce la probabilità
    predetta di "benign" (classe 1). Valida lunghezza e tipo prima di chiamare il modello -- come la lezione 7
    validava con Pydantic, qui a mano, senza quella dipendenza."""
    if not isinstance(valores, (list, tuple)):
        raise TypeError("valores deve essere una lista o tupla")
    if len(valores) != len(NOMBRES_COLUMNAS_CANCER):
        raise ValueError(f"ci si aspettavano {len(NOMBRES_COLUMNAS_CANCER)} valori, arrivati {len(valores)}")
    if not all(isinstance(v, (int, float)) for v in valores):
        raise TypeError("tutti i valori devono essere numerici")
    fila = np.array(valores).reshape(1, -1)
    probabilidad = float(pipeline_cancer.predict_proba(fila)[0, 1])
    return {"probabilidad_benigno": round(probabilidad, 4), "prediccion": int(probabilidad >= 0.5)}


ejemplo_valido = list(Xc_test[0])
resultado_prediccion = predecir(ejemplo_valido)
print("Predizione di esempio:", resultado_prediccion)

try:
    predecir([1.0, 2.0])  # lunghezza sbagliata -- deve fallire
    fallo_validacion = False
except ValueError:
    fallo_validacion = True
assert fallo_validacion
print("predecir() rifiuta correttamente un input con lunghezza sbagliata.")

In [ ]:
# 4.4 Un controllo di monitoraggio/soglia (lezione 10): confrontare l'accuratezza di un lotto recente contro
# una soglia minima, avvisando se scende sotto
UMBRAL_EXACTITUD_MINIMA = 0.90


def revisar_umbral_calidad(exactitud_actual, umbral=UMBRAL_EXACTITUD_MINIMA):
    """Stessa logica di detectar_incidentes della lezione 10, applicata a una singola metrica di modello."""
    return exactitud_actual < umbral


hay_alerta = revisar_umbral_calidad(EXACTITUD_CANCER)
print("Accuratezza attuale:", EXACTITUD_CANCER, "| soglia minima:", UMBRAL_EXACTITUD_MINIMA, "| allerta?:", hay_alerta)
assert hay_alerta is False

# Simuliamo un lotto degradato (accuratezza abbassata di proposito) per verificare che l'allerta SI attivi
assert revisar_umbral_calidad(0.80) is True
print("La funzione rileva correttamente sia il caso normale sia quello degradato.")

HERRAMIENTAS_REALES = [
    "DVC (lezione 3): versionare questo dataset e le sue future varianti, con un remoto proprio.",
    "MLflow (lezioni 4 e 8): registrare ogni addestramento (seed, iperparametri, metriche) e il modello con alias candidato/produzione/archiviato.",
    "pandera (lezione 6): uno schema di dati per le 30 colonne numeriche (intervalli, nulli).",
    "FastAPI (lezione 7): incapsulare predecir() in un endpoint reale quando il progetto passa in produzione.",
    "Docker + GitHub Actions (lezione 8): pacchettizzare il servizio e automatizzare i test prima di distribuire.",
    "Evidently (lezione 11): sorvegliare il drift delle 30 colonne se la fonte di dati cambia nel tempo.",
]
print("Strumenti che si userebbero qui in un progetto reale, SENZA reimplementarli (il criterio conta più che ripetere il codice):")
for herramienta in HERRAMIENTAS_REALES:
    print("-", herramienta)

**Cosa osservi?** Con un dataset diverso e un modello diverso, si ripetono esattamente gli stessi CINQUE tipi di decisione di tutto il corso: config + seed esplicita (lezione 2), test di dati e di modello (lezione 6), una funzione di servizio con validazione minima (lezione 7) e un controllo di soglia di monitoraggio (lezione 10) -- senza rimontare DVC, MLflow, Docker né FastAPI per questo caso piccolo, ma sapendo esattamente dove incastrerebbe ciascuno.

---
## Pulizia finale della cartella di lavoro

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo

- Un registro di audit leggibile si costruisce unendo eventi di dati (DVC), esperimenti (MLflow) e promozioni (registro dei modelli), e permette di verificare principi di controllo degli accessi come la separazione dei compiti.
- Il costo di un sistema di ML in produzione ha almeno tre componenti con formule distinte: addestramento (una volta), inferenza (per volume) e storage (per dimensione) -- tutte calcolabili in Python con costanti dichiarate.
- L'LLMOps condivide con l'MLOps classico il bisogno di tracciare, versionare e sorvegliare, ma cambia l'unità di costo (token, non ore di calcolo) e l'unità di tracciamento (una catena di passi, con `mlflow.trace`, non un singolo run).
- Lo stesso criterio di tutto il corso (config esplicita, test minimi, una funzione di servizio validata, una soglia di monitoraggio) si applica altrettanto bene a un progetto nuovo e piccolo quanto al progetto churn delle lezioni 2-11.